In [ ]:
# 1. Dọn dẹp các thư viện cũ để tránh xung đột
!pip uninstall -y transformers huggingface_hub peft bitsandbytes

# 2. Cài đặt các thư viện mới nhất
!pip install -q torch torchvision torchaudio accelerate tqdm
!pip install -q --upgrade transformers huggingface_hub peft bitsandbytes

In [ ]:
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN FILE & MODEL
# ==========================================
INPUT_PATH = "/kaggle/input/datasets/nhimchauphii/nhimhoang/test_part1_200.json"   # File Đề thi (Golden Dataset) 200 câu
OUTPUT_PATH = "t4_gemma_outputs_part1.json"                                       # File kết quả để đem đi chấm điểm benchmark

BASE_MODEL_ID = "unsloth/gemma-4-12b-it" 
LORA_MODEL_ID = "Nhat-Quang/outfitmatch-stylist-final-gemma4-12b-it-lora"

In [ ]:
import json
import time
import os
import gc
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ==========================================
# 1. KHỞI TẠO VÀ TẢI MODEL TRÊN KAGGLE (T4 x2)
# ==========================================
print("🔄 Đang cấu hình và tải Base Model ở chế độ 8-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto" # Tự động phân bổ RAM qua 2 card T4
)

print(f"Đang đắp LoRA adapter ({LORA_MODEL_ID}) lên Base Model...")
model = PeftModel.from_pretrained(base_model, LORA_MODEL_ID)
model.eval() # Chuyển sang chế độ suy luận (Inference Mode)
print("Tải Model thành công!\n")

In [ ]:
# ==========================================
# 2. HÀM TÌM KIẾM TÀI LIỆU (RETRIEVAL)
# ==========================================
def retrieve_documents(item):
    """
    Nhiệm vụ: Lấy trực tiếp tập ngữ cảnh chuẩn đã được gán nhãn sẵn trong file test.
    """
    return item.get("contexts", [])

In [ ]:
# ==========================================
# 3. HÀM SINH CÂU TRẢ LỜI (GENERATION)
# ==========================================
def generate_answer(question: str, retrieved_contexts: list) -> str:
    """
    Lấy câu hỏi và tài liệu ghép thành Prompt, ép model trả lời trung thực.
    """
    if isinstance(retrieved_contexts, list):
        context_text = "\n- ".join([str(c) for c in retrieved_contexts])
    else:
        context_text = str(retrieved_contexts)
        
    if not context_text.strip():
        return "Tôi không tìm thấy thông tin."
    
    # Định dạng tin nhắn chuẩn sử dụng Chat Template của Gemma
    messages = [
        {
            "role": "system", 
            "content": "Bạn là chuyên gia tư vấn thời trang. Hãy trả lời câu hỏi của người dùng CHỈ DỰA TRÊN tài liệu được cung cấp. Nếu tài liệu không chứa thông tin để trả lời, hãy nói chính xác: 'Tôi không tìm thấy thông tin'."
        },
        {
            "role": "user", 
            "content": f"--- TÀI LIỆU THAM KHẢO ---\n- {context_text}\n--------------------------\nCâu hỏi: {question}"
        }
    ]
    
    # Chuyển đổi qua chat template của Gemma
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    # Cấu hình Stop Tokens để tránh lỗi lặp / sinh hội thoại vô tận
    eos_ids = [tokenizer.eos_token_id]
    for token in ["<end_of_turn>", "<eos>"]:
        token_id = tokenizer.convert_tokens_to_ids(token)
        if token_id is not None and token_id not in eos_ids and not isinstance(token_id, list):
            eos_ids.append(token_id)
            
    input_length = inputs["input_ids"].shape[1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512, 
            temperature=0.01,   # Gần bằng 0 để tối ưu tính trung thực
            do_sample=True,
            top_p=0.9,
            use_cache=True,
            eos_token_id=eos_ids
        )
        
    # Cắt bỏ phần prompt đầu vào, chỉ lấy đúng câu trả lời mới sinh ra
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
# ==========================================
# 4. VÒNG LẶP CHÍNH (Ô SỐ 7)
# ==========================================
def run_evaluation_pipeline(input_file: str, output_file: str, save_interval: int = 10, limit: int = None):
    dataset = []
    
    # 1. Đọc dữ liệu (Checkpoint hoặc Khởi tạo mới)
    if os.path.exists(output_file):
        print(f"🔄 Đang tải Checkpoint từ: {output_file}...")
        with open(output_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    else:
        print(f"📖 Đang đọc file gốc: {input_file}...")
        with open(input_file, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
            
    if limit is not None:
        print(f"⚠️ Giới hạn xử lý {limit} câu đầu tiên để test.")
        dataset = dataset[:limit]
            
    uncompleted_items = [item for item in dataset if "answer" not in item]
    completed_samples = len(dataset) - len(uncompleted_items)
    
    print(f"🚀 Tiến trình: {completed_samples}/{len(dataset)} câu đã hoàn thành.")
    
    # 2. Chạy vòng lặp
    for idx, item in enumerate(tqdm(
        uncompleted_items, 
        desc="Tiến trình làm bài", 
        initial=completed_samples, 
        total=len(dataset)
    )):
        question = item.get("question", "")
        
        # Bốc context có sẵn
        retrieved = retrieve_documents(item)
        item["retrieved_contexts"] = retrieved
        
        # Gọi model
        try:
            item["answer"] = generate_answer(question, retrieved)
        except Exception as e:
            item["answer"] = f"Lỗi sinh text: {str(e)}"
        
        # 3. Lưu checkpoint sau mỗi save_interval câu
        if (idx + 1) % save_interval == 0 or (idx + 1) == len(uncompleted_items):
            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(dataset, f, ensure_ascii=False, indent=2)
                
        # Dọn rác bộ nhớ sau mỗi câu hỏi để tránh lỗi OOM GPU
        gc.collect()
        torch.cuda.empty_cache()
                
    print(f"\n🎉 Hoàn thành! Kết quả lưu tại: {output_file}")

In [ ]:
# Gọi hàm để bắt đầu chạy vòng lặp 5 câu
run_evaluation_pipeline(INPUT_PATH, OUTPUT_PATH, limit=5)